# Proyecto Etapa 2. Selección de técnica de muestreo para la construcción de muestra inicial

**Instituto Tecnológico y de Estudios Superiores de Monterrey**  
**Maestría en Inteligencia Artificial Aplicada**  
**Análisis de grandes volúmenes de datos**  
**Docente:** Dr. Iván Olmos Pineda

**Equipo 37:**
* González Pérez, Mario Alberto — A01840153
* Palma Palacios, Christian Ricardo — A01686081
* Salas Fuentes, Abraham Israel — A01840087
* Vega Martínez, Ángel — A01377304

**Fecha:** 17 de mayo de 2026

## 1. Descripción del dataset

El conjunto de datos `Microsoft_GUIDE_Train.csv` contiene eventos, alertas e incidentes de ciberseguridad provenientes de múltiples organizaciones.

**Características principales:**
- Más de 9.5 millones de registros.
- 45 variables heterogéneas.
- Datos reales de operaciones SOC (Security Operations Center).
- Clasificación supervisada de incidentes.

**Variable objetivo:**
- `IncidentGrade`

**Variables de caracterización seleccionadas para particionamiento:**
- `IncidentGrade` (Estratificación primaria)
- `Category` (Estratificación secundaria)
- `EntityType` (Estratificación terciaria)
- `EvidenceRole` (Estratificación operacional)

## 2. Configuración del entorno de PySpark en Google Colab

A continuación, se instalan las dependencias necesarias para ejecutar PySpark en el entorno de Google Colab.

In [1]:
# Instalación de Java y descarga de Spark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

In [2]:
# Configuración de variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

In [3]:
# Inicialización de la sesión de Spark
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BigData_Particionamiento") \
    .getOrCreate()

# Habilitar formato de salida mejorado para DataFrames
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

## 3. Carga de datos desde Google Drive

Se monta la unidad de Google Drive y se carga el archivo CSV en un DataFrame de PySpark.

In [4]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Definir ruta del archivo
file_path = "/content/drive/MyDrive/Colab Notebooks/MNA/BigData/ProyectoEtapa1/Microsoft_GUIDE_Train.csv"

Mounted at /content/drive


In [5]:
# Cargar el dataset
df = spark.read.csv(file_path, header=True, sep=",", inferSchema=True)

# Mostrar las primeras 5 filas para verificar la carga
df.limit(5).toPandas()

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04 06:05:15,7,6,InitialAccess,None,TruePositive,...,None,None,5,66,None,None,None,31,6,3
1,455266534868,88,326,210035,2024-06-14 03:01:25,58,43,Exfiltration,None,FalsePositive,...,None,None,5,66,None,None,None,242,1445,10630
2,1056561957389,809,58352,712507,2024-06-13 04:52:55,423,298,InitialAccess,T1189,FalsePositive,...,None,None,5,66,None,Suspicious,Suspicious,242,1445,10630
3,1279900258736,92,32992,774301,2024-06-10 16:39:36,2,2,CommandAndControl,None,BenignPositive,...,None,None,5,66,None,Suspicious,Suspicious,242,1445,10630
4,214748368522,148,4359,188041,2024-06-15 01:08:07,9,74,Execution,None,TruePositive,...,None,None,5,66,None,None,None,242,1445,10630


## 4. Preprocesamiento básico

Se eliminan registros nulos en la variable objetivo y registros duplicados para asegurar la calidad de los datos antes del particionamiento.

In [6]:
from pyspark.sql.functions import col, when, count, lit

# Eliminar registros sin variable objetivo
df = df.filter(col("IncidentGrade").isNotNull())

# Eliminar duplicados
df = df.dropDuplicates()

# Obtener cantidad total de registros limpios
total_records = df.count()
print(f"Cantidad total de registros limpios: {total_records:,}")

Cantidad total de registros limpios: 9,442,956


## 5. Construcción de variables agrupadas para particionamiento

Para evitar una explosión combinatoria y estratos demasiado pequeños, se aplica una estrategia de agrupación *Top-N* sobre las variables categóricas de mayor cardinalidad.

In [7]:
# Agrupación de la variable Category (Top 7)
top_categories = [
    "InitialAccess",
    "Exfiltration",
    "SuspiciousActivity",
    "CommandAndControl",
    "Impact",
    "CredentialAccess",
    "Execution"
]

df = df.withColumn(
    "CategoryGroup",
    when(col("Category").isin(top_categories), col("Category"))
    .otherwise("OtherCategory")
)

# Agrupación de la variable EntityType (Top 2)
df = df.withColumn(
    "EntityTypeGroup",
    when(col("EntityType") == "Ip", "Ip")
    .when(col("EntityType") == "User", "User")
    .otherwise("Other")
)

## 6. Generación y análisis de particiones

Se definen las columnas que conformarán las reglas de particionamiento y se calculan las combinaciones observadas empíricamente en el conjunto de datos.

In [8]:
# Definir columnas de particionamiento
partition_columns = [
    "IncidentGrade",
    "CategoryGroup",
    "EntityTypeGroup",
    "EvidenceRole"
]

# Calcular frecuencias y probabilidades empíricas de cada partición de forma eficiente
partition_stats = df.groupBy(*partition_columns) \
    .agg(count("*").alias("RecordCount")) \
    .withColumn("EmpiricalProbability_Pct", (col("RecordCount") / lit(total_records)) * 100) \
    .orderBy(col("RecordCount").desc())

total_partitions = partition_stats.count()
print(f"Número total de particiones observadas: {total_partitions}")
print("\nTop 10 particiones más frecuentes:")
partition_stats.limit(10).toPandas()

Número total de particiones observadas: 115

Top 10 particiones más frecuentes:


,IncidentGrade,CategoryGroup,EntityTypeGroup,EvidenceRole,RecordCount,EmpiricalProbability_Pct
0,TruePositive,InitialAccess,Other,Related,920775,9.750919
1,BenignPositive,Exfiltration,Other,Impacted,823594,8.721782
2,TruePositive,InitialAccess,Ip,Related,602878,6.384420
3,TruePositive,InitialAccess,User,Impacted,580458,6.146995
4,BenignPositive,InitialAccess,Other,Related,555916,5.887097
5,FalsePositive,InitialAccess,Other,Related,370120,3.919535
6,BenignPositive,CommandAndControl,Other,Related,308266,3.264507
7,BenignPositive,Impact,Ip,Related,280328,2.968647
8,BenignPositive,InitialAccess,Other,Impacted,264686,2.802999
9,FalsePositive,Exfiltration,Other,Impacted,255257,2.703147


### 6.1. Extracción de submuestras de prueba por partición

A continuación, se demuestra cómo extraer los registros correspondientes a una regla de particionamiento específica, cumpliendo con el requerimiento de filtrar y recuperar instancias.

In [9]:
# Ejemplo 1: BenignPositive + InitialAccess + Ip + Related
partition_1 = df.filter(
    (col("IncidentGrade") == "BenignPositive") &
    (col("CategoryGroup") == "InitialAccess") &
    (col("EntityTypeGroup") == "Ip") &
    (col("EvidenceRole") == "Related")
)

count_p1 = partition_1.count()
prob_p1 = (count_p1 / total_records) * 100
print(f"Partición 1 -> Registros: {count_p1:,} | Probabilidad empírica: {prob_p1:.4f}%")

# Ejemplo 2: TruePositive + Exfiltration + User + Impacted
partition_2 = df.filter(
    (col("IncidentGrade") == "TruePositive") &
    (col("CategoryGroup") == "Exfiltration") &
    (col("EntityTypeGroup") == "User") &
    (col("EvidenceRole") == "Impacted")
)

count_p2 = partition_2.count()
prob_p2 = (count_p2 / total_records) * 100
print(f"Partición 2 -> Registros: {count_p2:,} | Probabilidad empírica: {prob_p2:.4f}%")

Partición 1 -> Registros: 65,116 | Probabilidad empírica: 0.6896%
Partición 2 -> Registros: 19,927 | Probabilidad empírica: 0.2110%


## 7. Validación de la técnica de muestreo

Se verifica el correcto funcionamiento del código extrayendo una submuestra de prueba mediante **Muestreo Aleatorio Simple Sin Reemplazo** sobre una de las particiones, simulando la técnica que se aplicará para construir la muestra final.

In [10]:
# Extraer una submuestra del 10% de la Partición 1 sin reemplazo
sample_fraction = 0.10
seed_value = 42

sample_p1 = partition_1.sample(withReplacement=False, fraction=sample_fraction, seed=seed_value)

sample_count = sample_p1.count()
print(f"Registros en la partición original: {count_p1:,}")
print(f"Registros en la submuestra extraída (aprox. 10%): {sample_count:,}")

# Mostrar algunos registros de la submuestra
sample_p1.select("IncidentId", *partition_columns).limit(5).toPandas()

Registros en la partición original: 65,116
Registros en la submuestra extraída (aprox. 10%): 6,541


,IncidentId,IncidentGrade,CategoryGroup,EntityTypeGroup,EvidenceRole
0,96065,BenignPositive,InitialAccess,Ip,Related
1,37444,BenignPositive,InitialAccess,Ip,Related
2,509885,BenignPositive,InitialAccess,Ip,Related
3,111619,BenignPositive,InitialAccess,Ip,Related
4,157225,BenignPositive,InitialAccess,Ip,Related


## 8. Conclusión

El proceso implementado permite:
1. Dividir eficientemente el conjunto de datos original en estratos homogéneos.
2. Preservar la estructura estadística de la población mediante la identificación de probabilidades empíricas.
3. Extraer submuestras representativas utilizando muestreo aleatorio simple sin reemplazo dentro de cada estrato.

El enfoque basado en PySpark demuestra ser altamente apropiado para escenarios de procesamiento masivo (*Big Data*), permitiendo manejar la alta cardinalidad y el volumen masivo de registros de manera distribuida y eficiente.

---
**Repositorio del proyecto:**
[GitHub - Proyecto Etapa 2](https://github.com/cpalma-dev/BigData/blob/main/Proyecto_Etapa_2.1.ipynb)